# JSON-to-RDF (Only Entity) Converter

In [1]:
def generate_entity_rdf_anner(file, labeling_schema):

    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    xsd_date = '^^xsd:date'
    xsd_string = '^^xsd:string'
    xsd_non_neg_int = '^^xsd:nonNegativeInteger'
        
    rdf = ''
    labels_found = {}
    annotators = {
        'model': [],
        'human': []
    }

    schema_name = labeling_schema['name']
    schema_labels = labeling_schema['labels']
    
    for paragraph in data['annotations']:
        paragraph_id = paragraph[0]
        paragraph_text = paragraph[1]
        entity_ids = []
    
        # join entity ids in a single string
        for entity in paragraph[2]['entities']:
            entity_id = entity[0]
            entity_ids.append(f'data:{entity_id}')

        if bool(entity_ids):
            entity_ids_joined = ', '.join(entity_ids)
            
            rdf += f"data:{paragraph_id} onner:directlyContainsLabeledTerm {entity_ids_joined} .\n\n"
        
            # access all information of annotations
            for entity in paragraph[2]['entities']:
                entity_id = entity[0]
                start = int(entity[1])
                entity_length = start + int(entity[2])
                end = start + entity_length
                entity_text = paragraph_text[start:end]
                status_ids = []
                
                rdf += f"data:{entity[0]} rdf:type onner:LabeledTerm ;\n"    # deal with atomic and compound terms
                rdf += f"onner:labeledTermText '{entity_text}'{xsd_string} ;\n"
                rdf += f"onner:offset '{start}'{xsd_non_neg_int} ;\n"
                rdf += f"onner:length '{entity_length}'{xsd_non_neg_int} ;\n"
                rdf += f"onner:labeledTermDirectlyContainedBy data:{paragraph_id} ;\n"
        
                # join all status ids of an annotation in a single string
                for status_block in entity[3]:
                    status = status_block[0]
                    status_id = f'{status}_{entity_id}'
                    status_ids.append(status_id)
        
                status_ids_joined = ', '.join(status_ids)
                
                rdf += f"onner:hasLabeledTermStatus data:{status_ids_joined} .\n\n"
        
                # access the status block
                for status_block in entity[3]:
                    status = status_block[0]
                    status_id = f'{status}_{entity_id}'
        
                    label = status_block[1]
                    label_number = schema_labels[label]
                    
                    status_date = status_block[2]
                    status_assigner = status_block[3]
        
                    # store all unique labels found in a file
                    if label not in labels_found:
                        labels_found.update({label: label_number})
        
                    # store all unique annotator found in a file
                    if 'Model_' in status_assigner:
                        if status_assigner not in annotators['model']:
                            annotators['model'].append(status_assigner)
                    else:
                        if status_assigner not in annotators['human']:
                            annotators['human'].append(status_assigner)
        
                    rdf += f"data:{status_id} rdf:type onner:{status}Status ;\n"
                    rdf += f"onner:statusAssignmentDate '{status_date}'{xsd_date} ;\n"
                    rdf += f"onner:statusAssignedBy data:{status_assigner} ;\n"
                    rdf += f"onner:hasLabeledTermLabel data:{schema_name}_Label{label_number} .\n\n"

        else:    # check this condition with zero-entity paragraphs
            rdf += f"data:{paragraph_id} onner:directlyContainsLabeledTerm data:NoLabeledTerm .\n\n"
            
    for key, value in labels_found.items():
        rdf += f"data:{schema_name}_Label{value} rdf:type onner:Label ;\n"
        rdf += f"onner:fromLabelingSchema data:Labeling_Schema ;\n"
        rdf += f"onner:labelText '{key}'{xsd_string} .\n\n"
    
    rdf += f"data:{schema_name}_Labels rdf:type onner:LabelingSchema ;\n"
    rdf += f"onner:schemaName '{schema_name}'{xsd_string} .\n\n"
    
    for model in annotators['model']:
        version = model.split('_')[-1]
        rdf += f"data:{model} rdf:type onner:NER_System ;\n"
        rdf += f"onner:systemVersion '{version}'{xsd_string} .\n\n"
    
    for name in annotators['human']:
        rdf += f"data:Person_{name} rdf:type onner:Person ;\n"
        rdf += f"onner:personName '{name}'{xsd_string} .\n\n"
    
    return rdf



In [2]:
# test cell: function's output check
file = '/home/umayer/Work/research/ner_data/annotation_reviewed/Koshkava_2014_2of8_Torsten.json'
labeling_schema = {
    'name': 'CelloGraph',
    'labels': {
        'CHEM_ENT':         1, 
        'MAT_ENT_STRUCT':   2, 
        'MAT_ENT_UNSTRUCT': 3,
        'PROPERTY':         4,
        'END_USE':          5,
        'PROCESS':          6,
        'EQUIPMENT':        7,
        'MEASUREMENT':      8,
        'ABBREVIATION':     9        
    }
}

rdf = generate_entity_rdf_anner(file, labeling_schema)
print(rdf)

data:10.1016_j.powtec.2014.04.016_S2.1_P1 onner:directlyContainsLabeledTerm data:10.1016_j.powtec.2014.04.016_S2.1_P1_E1_Rv1, data:10.1016_j.powtec.2014.04.016_S2.1_P1_E2_Rv1, data:10.1016_j.powtec.2014.04.016_S2.1_P1_E3_Rv1, data:10.1016_j.powtec.2014.04.016_S2.1_P1_E1, data:10.1016_j.powtec.2014.04.016_S2.1_P1_E4_Rv1, data:10.1016_j.powtec.2014.04.016_S2.1_P1_E5_Rv1, data:10.1016_j.powtec.2014.04.016_S2.1_P1_E2, data:10.1016_j.powtec.2014.04.016_S2.1_P1_E3, data:10.1016_j.powtec.2014.04.016_S2.1_P1_E6_Rv1, data:10.1016_j.powtec.2014.04.016_S2.1_P1_E7_Rv1, data:10.1016_j.powtec.2014.04.016_S2.1_P1_E4, data:10.1016_j.powtec.2014.04.016_S2.1_P1_E5, data:10.1016_j.powtec.2014.04.016_S2.1_P1_E8_Rv1, data:10.1016_j.powtec.2014.04.016_S2.1_P1_E6, data:10.1016_j.powtec.2014.04.016_S2.1_P1_E9_Rv1, data:10.1016_j.powtec.2014.04.016_S2.1_P1_E7, data:10.1016_j.powtec.2014.04.016_S2.1_P1_E10_Rv1, data:10.1016_j.powtec.2014.04.016_S2.1_P1_E8, data:10.1016_j.powtec.2014.04.016_S2.1_P1_E11_Rv1, data